In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Annotated
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import os


In [4]:
# 상태 정의
class AgentState(TypedDict):
    query: str
    context: List[str]
    response: str
    retrieve_count: int

In [5]:
# 허깅페이스 모델 초기화
model_name = "cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
# 파이프라인 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=250
)

# LangChain HuggingFacePipeline 생성
llm = HuggingFacePipeline(pipeline=pipe)

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [7]:
#임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")


/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
# FAISS 벡터 저장소 초기화 또는 로드
index_name = "faiss_index"
if os.path.exists(f"{index_name}.faiss"):
    vectorstore = FAISS.load_local(index_name, embeddings)
else:
    # 초기 문서로 벡터 저장소 생성
    documents = [
        "The sky is blue.",
        "Grass is green.",
        "The sun is yellow."
    ]
    vectorstore = FAISS.from_texts(documents, embeddings)
    vectorstore.save_local(index_name)

In [9]:
def retrieve(state: AgentState):
    try:
        query = state['query']
        docs = vectorstore.similarity_search(query, k=3)
        state['context'] = [doc.page_content for doc in docs]
        state['retrieve_count'] = state.get('retrieve_count', 0) + 1
    except Exception as e:
        print(f"Error in retrieve: {e}")
        state['context'] = []
    return state

In [10]:
def generate_answer(state: AgentState):
    context = "\n".join(state['context'])
    prompt = f"Query: {state['query']}\nContext: {context}\nAnswer:"
    prompt = tokenizer.decode(tokenizer.encode(prompt)[:1024])  # 토큰 수 제한
    response = llm(prompt)
    state['response'] = response[0]['generated_text'].split("Answer:")[-1].strip()
    return state

In [11]:
def check_relevance(state: AgentState):
    relevance_prompt = f"Query: {state['query']}\nResponse: {state['response']}\nIs this response relevant and complete? Answer YES or NO."
    relevance_check = llm(relevance_prompt)[0]['generated_text']
    return "continue" if "NO" in relevance_check.upper() else "end"

In [12]:
# 그래프 구성
workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate_answer)
workflow.add_node("check_relevance", check_relevance)

In [13]:
# 시작점 설정
workflow.set_entry_point("retrieve")

# 엣지 설정
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "check_relevance")
workflow.add_edge("check_relevance", "generate", condition=lambda x: x == "continue")
workflow.add_edge("check_relevance", END, condition=lambda x: x == "end")

TypeError: StateGraph.add_edge() got an unexpected keyword argument 'condition'

In [ ]:
# 최대 반복 횟수 설정
def check_max_iterations(state):
    return state['retrieve_count'] >= 3

workflow.add_edge("check_relevance", END, condition=check_max_iterations)

In [ ]:
# 그래프 컴파일
app = workflow.compile()

In [ ]:
# 실행 함수
def process_query(query: str):
    inputs = {"query": query, "context": [], "response": "", "retrieve_count": 0}
    result = app.invoke(inputs)
    return result['response']

In [ ]:
# 사용 예
if __name__ == "__main__":
    documents = ["The sky is blue.", "Grass is green.", "The sun is yellow."]
    vectorstore.add_texts(documents)
    query = "What color is the sky?"
    answer = process_query(query)
    print(f"Query: {query}")
    print(f"Answer: {answer}")